# 05 - Timestamp-aware context fusion and temporal events (V2)

Frame/shot/event docs, ASR-OCR-object joins, scene facets, `temporal-edges.parquet`, and provenance fields.

**Execution model:** the notebook is a thin wrapper around the versioned script in `notebooks_V2/scripts/`. The cell is safe to rerun; completed videos are skipped by `_SUCCESS.json`. `RUN_ARGS` below defaults to a **full run** (no `--max-videos`/`--max-frames` cap). Swap in the smaller sample list first if you want to sanity-check on 1 video before spending full GPU time.


In [ ]:
%pip install -q pandas pyarrow pillow
# The full run below enables Qwen2.5-VL (OptionalVLM); the smoke-test sample list
# disables it, so this install only matters for the full run. Bump transformers
# if Qwen2_5_VLForConditionalGeneration fails to import.
%pip install -q "transformers>=4.49" accelerate


In [ ]:
from __future__ import annotations
import json
import runpy
import sys
from pathlib import Path

SCRIPT_NAME = '05_context_fusion.py'
RUN_ARGS = ["--enable-vlm"]

def locate_v2_root() -> Path:
    candidates = [
        Path.cwd() / "pipelines" / "kaggle" / "notebooks_V2",
        Path.cwd() / "notebooks_V2",
        Path("/kaggle/working/notebooks_V2"),
    ]
    candidates.extend(path.parent.parent for path in Path("/kaggle/input").rglob("v2_runtime.py"))
    for candidate in candidates:
        if (candidate / "scripts" / SCRIPT_NAME).exists():
            return candidate
    raise FileNotFoundError("Attach the complete notebooks_V2 directory or run from the repository root")

V2_ROOT = locate_v2_root()
SCRIPT = V2_ROOT / "scripts" / SCRIPT_NAME
sys.path.insert(0, str(V2_ROOT / "lib"))
print("V2 root:", V2_ROOT)
print("Script:", SCRIPT)


In [ ]:
# Full run (RUN_ARGS above = ["--enable-vlm"]). To sample-test first, set RUN_ARGS = ["--max-videos", "1", "--max-shots", "20", "--no-enable-vlm"] instead.
sys.argv = [str(SCRIPT), *RUN_ARGS]
runpy.run_path(str(SCRIPT), run_name="__main__")


In [ ]:
from pathlib import Path
import json

output_root = Path("/kaggle/working/aic-v2")
markers = list(output_root.rglob("_SUCCESS.json")) if output_root.exists() else []
print("Success markers:", len(markers))
for marker in markers[-10:]:
    try:
        payload = json.loads(marker.read_text(encoding="utf-8"))
        print(marker, payload.get("stage"), payload.get("video_id"), payload.get("success"))
    except Exception as exc:
        print("Invalid marker:", marker, exc)
